# ch04 Bonus 03：多头潜在注意力（MLA）

> 对照官方 `rasbt/LLMs-from-scratch` ch04/05_mla
> **参考真实模型**：DeepSeek-V2 / DeepSeek-V3

## 一句话

把 K/V 压缩到一个**低秩潜变量** $c_{kv}$，KV 缓存只存 $c_{kv}$ 而非完整的 K/V，显存大幅下降（DeepSeek 实测省约 93%）。

## 为什么需要 MLA

GQA 通过分组减少 KV 头数，但每组仍要存完整的 `head_dim` 维。MLA 更激进：用一个**下投影矩阵** $W^{DKV}$ 把 `d_model` 压成 `d_compress`（如 768→192），只缓存压缩后的 $c_{kv}$；推理时再用**上投影** $W^{UKV}$ 还原成 K/V。

| 方案 | 每 token 缓存维度 | 省显存 |
|------|----------------|-------|
| MHA（12头）  | 2 × 768 = 1536 | 基准 |
| GQA（2组）   | 2 × 128 = 256  | 83% |
| **MLA**      | **192**        | **88%** |

> MLA 的压缩是有损的，但因低秩假设 + 训练，质量损失极小，DeepSeek-V3 在长上下文上表现优异。

## 核心改造

标准注意力：`K = W_K(x)`, `V = W_V(x)`。
MLA：`c_kv = W_DKV(x)`（下投影）→ `K,V = W_UKV(c_kv)`（上投影）。缓存只存 `c_kv`。

In [ ]:
import torch
import torch.nn as nn


class MultiHeadLatentAttention(nn.Module):
    """多头潜在注意力（MLA）教学版。

    将 K/V 压缩到低秩潜变量 c_kv，KV 缓存只存 c_kv。
    真实 DeepSeek-V2 还会用 RoPE 解耦，这里取教学简化版。
    """

    def __init__(self, d_in, d_out, context_length, num_heads, kv_compress_dim,
                 dropout=0.0):
        super().__init__()
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.kv_compress_dim = kv_compress_dim

        # 下投影：d_model → d_compress（这就是 KV 缓存的全部内容）
        self.W_dkv = nn.Linear(d_in, kv_compress_dim, bias=False)
        # 上投影：d_compress → d_out（还原出 K 和 V）
        self.W_ukv = nn.Linear(kv_compress_dim, d_out, bias=False)
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool(),
        )

    def forward(self, x):
        b, n, _ = x.shape
        c_kv = self.W_dkv(x)                  # [b, n, kv_compress_dim]  ← 缓存这个
        kv = self.W_ukv(c_kv)                 # [b, n, d_out]
        # 教学版：K 和 V 都从同一上投影出（真实实现会分两个或加 RoPE）
        k = kv.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        v = kv.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        q = self.W_query(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = q @ k.transpose(2, 3)
        mask_bool = self.mask.bool()[:n, :n]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.head_dim ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        out = (attn_weights @ v).transpose(1, 2).contiguous().view(b, n, self.d_out)
        return self.out_proj(out), c_kv       # 顺带返回压缩向量，供观察

## 2. 验证：输出形状 + 缓存省显存比例

In [ ]:
torch.manual_seed(123)
batch, seq, dim, n_heads = 2, 16, 768, 12
x = torch.randn(batch, seq, dim)

# 压缩到 192 维（DeepSeek-V2 用类似策略）
mla = MultiHeadLatentAttention(dim, dim, 1024, n_heads, kv_compress_dim=192)
out, c_kv = mla(x)
print(f"输出: {tuple(out.shape)}")
print(f"压缩潜变量 c_kv: {tuple(c_kv.shape)}")

mha_per_token = 2 * dim                      # 标准 MHA 每 token 缓存 K+V
mla_per_token = 192
print(f"\n每 token KV 缓存: MHA={mha_per_token}d  MLA={mla_per_token}d")
print(f"省显存: {100 * (1 - mla_per_token / mha_per_token):.0f}%")

## 3. 代价：多了上下投影的算力

MLA 用算力换显存：每次推理要多做一次下投影 + 上投影。但在长上下文场景，KV 缓存显存往往是瓶颈，这笔交易很划算。

---
> 📌 本 notebook 实现 MLA 教学版并验证缓存压缩比例。
> 完整 RoPE 解耦实现见官方 `ch04/05_mla`。